# QSAR Analysis — Biological Activity & Model Comparison

Visual analysis of log_LC50 distribution and model performance comparison.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import cross_val_predict, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor, BaggingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

## Part 1: Biological Activity Analysis

In [2]:
df = pd.read_csv('data/curated_data_rdkit.csv')
y = df['log_LC50'].values

print(f'Total compounds: {len(y)}')
print(f'Mean log_LC50: {y.mean():.3f} +/- {y.std():.3f}')
print(f'Median: {np.median(y):.3f}')
print(f'Range: [{y.min():.3f}, {y.max():.3f}]')
print(f'Skewness: {pd.Series(y).skew():.3f}')
print(f'Kurtosis: {pd.Series(y).kurtosis():.3f}')

Total compounds: 2303
Mean log_LC50: 0.866 +/- 1.330
Median: 0.894
Range: [-5.000, 4.763]
Skewness: -0.321
Kurtosis: 0.659


### Distribution of log_LC50

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(y, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Mean = {y.mean():.2f}')
axes[0].axvline(np.median(y), color='orange', linestyle='--', label=f'Median = {np.median(y):.2f}')
axes[0].set_xlabel('log_LC50')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of log_LC50')
axes[0].legend()

axes[1].boxplot(y, widths=0.5)
axes[1].set_ylabel('log_LC50')
axes[1].set_title('Box Plot')

from scipy import stats
stats.probplot(y, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot')
axes[2].get_lines()[0].set_markerfacecolor('steelblue')

plt.tight_layout()
plt.savefig('figures/abdek_figures/activity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Toxicity Categories

In [4]:
def toxicity_category(lc50):
    if lc50 < -1:
        return 'Very High Toxicity'
    elif lc50 < 0:
        return 'High Toxicity'
    elif lc50 < 1:
        return 'Moderate Toxicity'
    elif lc50 < 2:
        return 'Low Toxicity'
    else:
        return 'Very Low Toxicity'

df['toxicity_class'] = df['log_LC50'].apply(toxicity_category)
class_order = ['Very High Toxicity', 'High Toxicity', 'Moderate Toxicity', 'Low Toxicity', 'Very Low Toxicity']
counts = df['toxicity_class'].value_counts().reindex(class_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#4575b4']
bars = axes[0].bar(range(len(counts)), counts.values, color=colors, edgecolor='white')
axes[0].set_xticks(range(len(counts)))
axes[0].set_xticklabels(counts.index, rotation=30, ha='right')
axes[0].set_ylabel('Number of compounds')
axes[0].set_title('Compounds by Toxicity Category')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                 f'{val} ({val/len(y)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proportion by Toxicity Category')

plt.tight_layout()
plt.savefig('figures/abdek_figures/toxicity_categories.png', dpi=150, bbox_inches='tight')
plt.show()

print(counts.to_string())

toxicity_class
Very High Toxicity    171
High Toxicity         369
Moderate Toxicity     671
Low Toxicity          634
Very Low Toxicity     458


### Activity vs Measurement Count

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

high_n = df[df['n_measurements'] >= 10]
axes[0].scatter(df['n_measurements'], df['log_LC50'], alpha=0.3, s=10, label='All')
axes[0].scatter(high_n['n_measurements'], high_n['log_LC50'], alpha=0.7, s=30, color='red', label='>=10 measurements')
axes[0].set_xlabel('Number of measurements')
axes[0].set_ylabel('log_LC50')
axes[0].set_title('Activity vs Measurement Count')
axes[0].legend()

df['mw_bin'] = pd.cut(df['AMW'], bins=5)
df.boxplot(column='log_LC50', by='mw_bin', ax=axes[1])
axes[1].set_xlabel('Molecular Weight Range')
axes[1].set_ylabel('log_LC50')
axes[1].set_title('Activity by Molecular Weight')
axes[1].set_xticklabels([f'{int(b.left)}-{int(b.right)}' for b in df['mw_bin'].cat.categories], rotation=45, ha='right')
plt.suptitle('')

plt.tight_layout()
plt.savefig('figures/abdek_figures/activity_vs_measurements.png', dpi=150, bbox_inches='tight')
plt.show()

### Descriptor-Activity Correlations

In [6]:
desc_cols = ['SlogP', 'TPSA', 'AMW', 'NumRotatableBonds', 'NumRings', 'NumHBD', 'NumHBA', 'FractionCSP3']
corr = df[desc_cols + ['log_LC50']].corr()['log_LC50'].drop('log_LC50').sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_corr = ['#d73027' if v > 0 else '#4575b4' for v in corr.values]
bars = axes[0].barh(range(len(corr)), corr.values, color=colors_corr, edgecolor='white')
axes[0].set_yticks(range(len(corr)))
axes[0].set_yticklabels(corr.index)
axes[0].set_xlabel('Correlation with log_LC50')
axes[0].set_title('Descriptor-Activity Correlation')
axes[0].axvline(0, color='gray', linewidth=0.5)

best_desc = corr.index[0]
axes[1].scatter(df[best_desc], df['log_LC50'], alpha=0.3, s=10, color='steelblue')
z = np.polyfit(df[best_desc], df['log_LC50'], 1)
p = np.poly1d(z)
axes[1].plot(df[best_desc].sort_values(), p(df[best_desc].sort_values()), 'r--', linewidth=2)
axes[1].set_xlabel(best_desc)
axes[1].set_ylabel('log_LC50')
axes[1].set_title(f'{best_desc} vs log_LC50 (r={corr.iloc[0]:.3f})')

plt.tight_layout()
plt.savefig('figures/abdek_figures/descriptor_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlations with log_LC50:')
print(corr.to_string())

Correlations with log_LC50:
NumHBD               0.202107
TPSA                 0.158230
FractionCSP3         0.138428
NumHBA               0.056811
NumRotatableBonds   -0.029709
NumRings            -0.152030
AMW                 -0.209760
SlogP               -0.435293


## Part 2: Model Performance Comparison

In [7]:
mols = df['SMILES'].apply(Chem.MolFromSmiles)
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = np.array([gen.GetFingerprint(m) for m in mols])

meta_cols = ['CAS_str', 'Chemical_name', 'SMILES', 'mlecule_H_final', 'SMILES_curated']
exclude_cols = meta_cols + ['log_LC50', 'n_measurements', 'toxicity_class', 'mw_bin']
desc_cols_all = [c for c in df.columns if c not in exclude_cols]
X_combined = np.hstack([df[desc_cols_all].values, fps])
all_feature_names = desc_cols_all + [f'FP_{i}' for i in range(fps.shape[1])]

var_thresh = VarianceThreshold(threshold=0.01)
X_var = var_thresh.fit_transform(X_combined)
kept_mask = var_thresh.get_support()
kept_names = [all_feature_names[i] for i in range(len(all_feature_names)) if kept_mask[i]]

corr_matrix = pd.DataFrame(X_var).corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
keep_idx = [i for i in range(X_var.shape[1]) if i not in to_drop]
X_final = X_var[:, keep_idx]
final_names = [kept_names[i] for i in keep_idx]

print(f'Features: {X_final.shape[1]}, Samples: {X_final.shape[0]}')

Features: 513, Samples: 2303


In [8]:
models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
    'SVR': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    'PLS': PLSRegression(n_components=20),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    'Bagging': BaggingRegressor(n_estimators=100, max_samples=0.8, n_jobs=-1, random_state=42),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []
cv_preds = {}

for name, model in models.items():
    X = X_final.copy()
    if name in ['Ridge', 'Lasso', 'ElasticNet', 'SVR', 'KNN', 'PLS']:
        X = StandardScaler().fit_transform(X)
    y_pred = cross_val_predict(model, X, y, cv=cv)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    cv_results.append({'Model': name, 'R2': r2, 'RMSE': rmse, 'MAE': mae})
    cv_preds[name] = y_pred

cv_df = pd.DataFrame(cv_results).sort_values('R2', ascending=False).reset_index(drop=True)
print(cv_df.to_string(index=False))

           Model       R2     RMSE      MAE
        LightGBM 0.531846 0.909733 0.654759
         XGBoost 0.524626 0.916722 0.648061
    RandomForest 0.521533 0.919700 0.655443
      ExtraTrees 0.520398 0.920789 0.642282
GradientBoosting 0.519224 0.921916 0.654870
         Bagging 0.518372 0.922733 0.658807
             SVR 0.471913 0.966212 0.705420
           Lasso 0.399874 1.030010 0.761016
      ElasticNet 0.371344 1.054208 0.774068
        AdaBoost 0.348935 1.072833 0.817682
             KNN 0.293225 1.117791 0.839667
             PLS 0.246609 1.154065 0.840031
           Ridge 0.239168 1.159750 0.846004


### Model Comparison — Bar Charts

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_names = cv_df['Model'].values
x_pos = range(len(model_names))

colors_bar = plt.cm.viridis(np.linspace(0.2, 0.8, len(model_names)))
sorted_idx = cv_df['R2'].argsort()

for ax, metric, title in zip(axes, ['R2', 'RMSE', 'MAE'], ['R² Score', 'RMSE', 'MAE']):
    sorted_df = cv_df.sort_values(metric, ascending=(metric != 'R2'))
    c = ['#2ca02c' if v == cv_df[metric].max() else '#1f77b4' for v in sorted_df[metric]] if metric == 'R2' else ['#d73027' if v == cv_df[metric].min() else '#1f77b4' for v in sorted_df[metric]]
    ax.barh(range(len(sorted_df)), sorted_df[metric].values, color=c, edgecolor='white')
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels(sorted_df['Model'].values)
    ax.set_xlabel(metric)
    ax.set_title(title)
    for i, v in enumerate(sorted_df[metric].values):
        ax.text(v + (0.01 if metric == 'R2' else 0.01), i, f'{v:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('figures/abdek_figures/model_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### Parity Plots — All Models

In [10]:
top_models = cv_df.head(6)['Model'].values
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

y_min, y_max = y.min(), y.max()

for i, name in enumerate(top_models):
    ax = axes[i]
    ax.scatter(y, cv_preds[name], alpha=0.4, s=8, color='steelblue')
    ax.plot([y_min, y_max], [y_min, y_max], 'r--', linewidth=2, alpha=0.7)
    r2 = cv_df[cv_df['Model'] == name]['R2'].values[0]
    rmse = cv_df[cv_df['Model'] == name]['RMSE'].values[0]
    ax.set_xlabel('Actual log_LC50')
    ax.set_ylabel('Predicted log_LC50')
    ax.set_title(f'{name}\nR²={r2:.3f}, RMSE={rmse:.3f}')
    ax.set_xlim(y_min - 0.5, y_max + 0.5)
    ax.set_ylim(y_min - 0.5, y_max + 0.5)
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('figures/abdek_figures/parity_plots_cv.png', dpi=150, bbox_inches='tight')
plt.show()

### Residual Plots — All Models

In [11]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, name in enumerate(top_models):
    ax = axes[i]
    residuals = y - cv_preds[name]
    ax.scatter(cv_preds[name], residuals, alpha=0.4, s=8, color='steelblue')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axhline(y=residuals.std(), color='gray', linestyle=':', alpha=0.5)
    ax.axhline(y=-residuals.std(), color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Predicted log_LC50')
    ax.set_ylabel('Residual')
    ax.set_title(f'{name}\nStd residual = {residuals.std():.3f}')

plt.tight_layout()
plt.savefig('figures/abdek_figures/residual_plots_cv.png', dpi=150, bbox_inches='tight')
plt.show()

### Residual Distribution Histograms

In [12]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, name in enumerate(top_models):
    ax = axes[i]
    residuals = y - cv_preds[name]
    ax.hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Residual')
    ax.set_ylabel('Count')
    ax.set_title(f'{name}\nMean={residuals.mean():.3f}, Std={residuals.std():.3f}')

plt.tight_layout()
plt.savefig('figures/abdek_figures/residual_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

### Train/Test Split — Final Validation

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

test_results = []
test_preds = {}

for name, model in models.items():
    X_tr, X_te = X_train.copy(), X_test.copy()
    if name in ['Ridge', 'Lasso', 'ElasticNet', 'SVR', 'KNN', 'PLS']:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    test_r2 = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    test_results.append({'Model': name, 'Test_R2': test_r2, 'Test_RMSE': test_rmse, 'Test_MAE': test_mae})
    test_preds[name] = y_pred

test_df = pd.DataFrame(test_results).sort_values('Test_R2', ascending=False).reset_index(drop=True)
print(test_df.to_string(index=False))

           Model  Test_R2  Test_RMSE  Test_MAE
         XGBoost 0.509274   0.918020  0.642206
        LightGBM 0.507802   0.919396  0.650740
GradientBoosting 0.481918   0.943260  0.653537
    RandomForest 0.472964   0.951377  0.668893
         Bagging 0.468550   0.955352  0.675289
      ExtraTrees 0.468522   0.955377  0.650951
             SVR 0.444335   0.976875  0.707695
           Lasso 0.365192   1.044128  0.764937
      ElasticNet 0.341842   1.063158  0.776022
        AdaBoost 0.303504   1.093683  0.830056
             KNN 0.245861   1.138042  0.840211
             PLS 0.221490   1.156284  0.839466
           Ridge 0.207493   1.166632  0.851850


### Test Set Parity Plots

In [14]:
top_test = test_df.head(6)['Model'].values
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, name in enumerate(top_test):
    ax = axes[i]
    ax.scatter(y_test, test_preds[name], alpha=0.5, s=15, color='steelblue', edgecolors='none')
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, alpha=0.7)
    r2 = test_df[test_df['Model'] == name]['Test_R2'].values[0]
    rmse = test_df[test_df['Model'] == name]['Test_RMSE'].values[0]
    ax.set_xlabel('Actual log_LC50')
    ax.set_ylabel('Predicted log_LC50')
    ax.set_title(f'{name} (Test)\nR²={r2:.3f}, RMSE={rmse:.3f}')
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('figures/abdek_figures/parity_plots_test.png', dpi=150, bbox_inches='tight')
plt.show()

### CV vs Test Performance Comparison

In [15]:
merged = cv_df.merge(test_df, left_on='Model', right_on='Model')
merged['Gap'] = merged['R2'] - merged['Test_R2']
merged = merged.sort_values('Test_R2', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x = range(len(merged))
width = 0.35
axes[0].bar([i - width/2 for i in x], merged['R2'], width, label='CV R²', color='steelblue')
axes[0].bar([i + width/2 for i in x], merged['Test_R2'], width, label='Test R²', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(merged['Model'].values, rotation=45, ha='right')
axes[0].set_ylabel('R²')
axes[0].set_title('CV vs Test R²')
axes[0].legend()

colors_gap = ['#d73027' if v > 0.3 else '#fc8d59' if v > 0.15 else '#fee08b' for v in merged['Gap']]
axes[1].bar(x, merged['Gap'].values, color=colors_gap, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(merged['Model'].values, rotation=45, ha='right')
axes[1].set_ylabel('R² Gap (CV - Test)')
axes[1].set_title('Overfitting Gap')
axes[1].axhline(y=0, color='gray', linewidth=0.5)
for i, v in enumerate(merged['Gap'].values):
    axes[1].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('figures/abdek_figures/cv_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()

print(merged[['Model', 'R2', 'Test_R2', 'Gap']].to_string(index=False))

           Model       R2  Test_R2      Gap
         XGBoost 0.524626 0.509274 0.015352
        LightGBM 0.531846 0.507802 0.024045
GradientBoosting 0.519224 0.481918 0.037306
    RandomForest 0.521533 0.472964 0.048569
         Bagging 0.518372 0.468550 0.049822
      ExtraTrees 0.520398 0.468522 0.051876
             SVR 0.471913 0.444335 0.027578
           Lasso 0.399874 0.365192 0.034682
      ElasticNet 0.371344 0.341842 0.029502
        AdaBoost 0.348935 0.303504 0.045431
             KNN 0.293225 0.245861 0.047365
             PLS 0.246609 0.221490 0.025119
           Ridge 0.239168 0.207493 0.031675


### Summary Report

In [16]:
print('='*60)
print('QSAR MODELING SUMMARY')
print('='*60)
print(f'Dataset: {len(y)} compounds')
print(f'Features after filtering: {X_final.shape[1]}')
print()
print('Best model (Test set):', test_df.iloc[0]['Model'])
print(f'  Test R²:  {test_df.iloc[0]["Test_R2"]:.4f}')
print(f'  Test RMSE: {test_df.iloc[0]["Test_RMSE"]:.4f}')
print(f'  Test MAE:  {test_df.iloc[0]["Test_MAE"]:.4f}')
print()
print('All plots saved:')
for f in ['activity_distribution.png', 'toxicity_categories.png', 'activity_vs_measurements.png',
          'descriptor_correlations.png', 'model_comparison_bars.png', 'parity_plots_cv.png',
          'residual_plots_cv.png', 'residual_histograms.png', 'parity_plots_test.png', 'cv_vs_test.png']:
    print(f'  - {f}')

QSAR MODELING SUMMARY
Dataset: 2303 compounds
Features after filtering: 513

Best model (Test set): XGBoost
  Test R²:  0.5093
  Test RMSE: 0.9180
  Test MAE:  0.6422

All plots saved:
  - activity_distribution.png
  - toxicity_categories.png
  - activity_vs_measurements.png
  - descriptor_correlations.png
  - model_comparison_bars.png
  - parity_plots_cv.png
  - residual_plots_cv.png
  - residual_histograms.png
  - parity_plots_test.png
  - cv_vs_test.png
